# Blackboard

https://arxiv.org/pdf/2507.01701

## Board

In [1]:
from pydantic import BaseModel, Field, model_validator
from enum import Enum
import uuid

def get_id(n : int = 6):
    uuid4 = uuid.uuid4().hex[:n]
    return str(uuid4).replace('-', '')

class NoteStatus(Enum):
    ACTIVE='active'
    """Запись активна"""
    MARKED_FOR_DELETION='marked_for_deletion'
    """Помечена к удалению"""
    DELETED='deleted'
    """Заметка была удалена"""

class BaseNote(BaseModel):
    """
    Короткая заметка для добавления на доску
    """
    content : str = Field(max_length=2000, description='Содержимое записи')
    summary : str = Field(max_length=280, description='Суть записи в одном коротком предложении')
    keywords : list[str] = Field(min_length=1, max_length=5, description='Ключевые слова')

class Note(BaseNote):
    id : str = Field(default='', description='ID записи')
    # status : NoteStatus = Field(default=NoteStatus.ACTIVE, description='Статус записи')
    author_id : str = Field(description='ID автора записи')

    @model_validator(mode='after')
    def _validate_model(self):
        self.id = get_id()
        return self

class Board(BaseModel):
    question : str = Field(description='Вопрос пользователя')
    notes : list[BaseNote] = Field(default=[], description='Список записей')

    def get_board_notes(self, last_n : int | None = None) -> list[dict]:
        """
        Возвращает список актуальных записей на доске с краткой информацией.
        Если передан last_n, вернет последние last_n записей.
        """
        notes = [{
            'id': note.id,
            'summary': note.summary,
            'keywords': note.keywords
        } for note in self.notes]
        if last_n is not None:
            notes = notes[-last_n:]
        return notes
    
    def get_board_note(self, note_id : str) -> Note | None:
        """
        Возвращает запись с указанным id.
        Возвращает None, если запись не найдена.
        """
        notes = [n for n in self.notes if n.id == note_id]
        if len(notes) == 0:
            return None
        return notes[0]
    
    def add_note(self, note : BaseNote, author_id : str) -> str:
        """
        Добавляет запись на доску.
        Возвращает id записи.
        """
        note = Note(author_id=author_id, **note.model_dump())
        self.notes.append(note)
        return note.id
    
    def remove_note(self, note_id : str):
        self.notes = [n for n in self.notes if n.id != note_id]

## Agents

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from src import llm

In [3]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import BaseMessage

STRUCTURED_RESPONSE_KEY = 'structured_response'
MESSAGES_KEY = 'messages'

class Agent():

    def __init__(
        self,
        *args,
        id_ : str | None = None,
        tools : list | None = None,
        system_prompt : str | None = None,
        response_format : type[BaseModel] | None = None,
        checkpointer : InMemorySaver | None = None,
        summarization_tokens : int = 4000, 
        summarization_keep : int = 10,
        **kwargs
    ):
        self.id = id_ or get_id()
        
        self.tools = tools or []
        self.system_prompt = self._format_system_prompt(system_prompt)
        self.response_format = response_format
        self.checkpointer = checkpointer or InMemorySaver()

        summarization_middleware = SummarizationMiddleware(
            model=llm,
            trigger=("tokens", summarization_tokens),
            keep=("messages", summarization_keep)
        )
        
        self._agent = create_agent(
            *args,
            model=llm, 
            tools=tools, 
            response_format=response_format, 
            system_prompt=self.system_prompt,
            checkpointer=self.checkpointer,
            middleware=[summarization_middleware],
            **kwargs
        )

    def _format_system_prompt(self, system_prompt : str | None) -> str | None:
        return system_prompt

    def invoke(self, messages : BaseMessage, thread_id : str | None = None, force : bool = False):
        response = None

        def invoke():
            return self._agent.invoke(
                input={'messages': messages},
                config={"configurable": {"thread_id": thread_id or self.id}},
            )

        if force:
            while response is None:
                try:
                    response = invoke()
                except:
                    print('Maybe tool calls or something idk')
        else:
            response = invoke()
            
        if STRUCTURED_RESPONSE_KEY in response:
            return response[STRUCTURED_RESPONSE_KEY]
        return response[MESSAGES_KEY][-1].content
    
    @property
    def info(self):
        return {
            'id': self.id
        }
    
class Role(BaseModel):
    name : str = Field(description='Название роли')
    description : str = Field(max_length=140, description='Описание роли')
    
class RoleAgent(Agent):

    def __init__(self, role : Role, *args, **kwargs):
        self.role = role
        super().__init__(*args, **kwargs)

    def _format_system_prompt(self, system_prompt):
        if system_prompt is None:
            return system_prompt
        return system_prompt.format(**self.info)

    @property
    def info(self):
        return {
            **super().info,
            'role_name': self.role.name,
            'role_description': self.role.description
        }

### Generator

In [4]:
class GeneratorResponse(BaseModel):
    roles : list[Role] = Field(min_length=1, max_length=3, description='Список экспертных ролей')

_generator_prompt = """
Вам дан вопрос. Предоставьте мне список экспертных ролей, которые наиболее полезны для решения вопроса.
Вопрос:
{question}
"""

def create_generator_agent(board : Board) -> Agent:
    system_prompt = _generator_prompt.format(question=board.question)
    return Agent(system_prompt=system_prompt, response_format=GeneratorResponse)

### Expert

In [5]:
_id_prompt = """
Ваш личный id {id}.
"""
_system_prompt = """
Вы {role_name}, сотрудничающий с другими агентами для решения задачи.
Задача:
{question}
Существует общая доска, которую каждый из вас может читать и на которой может оставлять записи.
"""
_expert_prompt = """
Вы выдающийся специалист:
{role_name}
Описание:
{role_description}

Основываясь на ваших экспертных знаниях и текущем содержимом общей доски, решите задачу, изложите свои идеи и информацию, которую вы хотите записать на доску.
Совершенно не обязательно полностью соглашаться с точкой зрения, представленной на доске.
"""

def create_expert_agent(role : Role, board : Board, ):
    system_prompt = _id_prompt + (_system_prompt + _expert_prompt).format(
        question=board.question,
        role_name=role.name,
        role_description=role.description
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_board_notes),
            tool(board.get_board_note)  
        ],
        response_format=BaseNote,
    )
    return agent

### Decider

In [6]:
class DeciderResponse(BaseModel):
    content : BaseNote | str = Field(description='Запись для добавления на доску (BaseNote) ИЛИ окончательный ответ на вопрос (str)')

_decider_prompt = """
Ваша задача проанализировать текущее состояние общей доски и решить, достаточно ли у команды информации для получения окончательного ответа.
Если информации на доске достаточно для решения задачи, вы должны указать, что работа завершена, и предоставить окончательный ответ.
Если для получения ответа необходима дополнительная информация от других агентов, вы должны указать, что процесс следует продолжить.
"""

def create_decider_agent(board : Board):
    role = Role(
        name='Арбитр',
        description='Оценивает полноту информации. Выдает финальный ответ, либо инициирует продолжение обсуждения'
    )
    system_prompt = _id_prompt + (_system_prompt + _decider_prompt).format(
        role_name=role.name,
        question=board.question,
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_board_notes),
            tool(board.get_board_note)  
        ],
        response_format=DeciderResponse,
    )
    return agent

### Planner

In [7]:
_planner_prompt = """
Создайте план решения исходной задачи на основе текущего содержимого общей доски.
Опишите задачу своими словами, затем изложите пошаговый план её решения.
Если план уже существует на доске или задача достаточно проста для прямого решения, просто укажите, что нет необходимости декомпозировать задачи и вы ожидаете дополнительной информации.
Не решайте задачу. Предоставьте только план.
"""

def create_planner_agent(board : Board):
    role = Role(
        name='Планировщик',
        description='Разрабатывает пошаговый план решения задачи на основе содержимого доски'
    )
    system_prompt = _id_prompt + (_system_prompt + _planner_prompt).format(
        role_name=role.name,
        question=board.question,
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_board_notes),
            tool(board.get_board_note)  
        ],
        response_format=BaseNote,
    )
    return agent

### Critic

In [8]:
_critic_prompt = """
Проанализируйте записи на общей доске и определите любые бесполезные или избыточные.

Если вы обнаружите такие записи, опишите их и объясните, почему они должны быть удалены.
Если бесполезных записей нет, просто укажите, что бесполезных записей нет и вы ожидаете дополнительной информации.
"""

def create_critic_agent(board : Board):
    role = Role(
        name='Критик',
        description='Выявляет ошибочные или вводящие в заблуждение записи на доске'
    )
    system_prompt = _id_prompt + (_system_prompt + _critic_prompt).format(
        role_name=role.name,
        question=board.question,
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_board_notes),
            tool(board.get_board_note)  
        ],
        response_format=BaseNote,
    )
    return agent

### Cleaner

In [9]:
_cleaner_prompt = """
Проанализируйте записи на общей доске и определите любые бесполезные или избыточные.

Если вы обнаружите такие записи:
- Перечислите их
- Для каждого объясните, почему оно бесполезно или избыточно

Если бесполезных записей нет, просто укажите, что бесполезных записей нет и вы ожидаете дополнительной информации.
"""

class CleanerResponse(BaseModel):
    note : BaseNote = Field(description='Запись для добавления на доску')
    notes_ids : list[str] = Field(default=[], description='Список ID записей к удалению')

def create_cleaner_agent(board : Board):
    role = Role(
        name='Уборщик',
        description='Анализирует доску, выявляет и удаляет бесполезные или избыточные записи'
    )
    system_prompt = _id_prompt + (_system_prompt + _cleaner_prompt).format(
        role_name=role.name,
        question=board.question,
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_board_notes),
            tool(board.get_board_note)  
        ],
        response_format=CleanerResponse,
    )
    return agent

### Controller

In [10]:
class ControllerResponse(BaseModel):
    agents_ids : list[str] = Field(min_length=1, description='Упорядоченный список ID агентов')

_controller_prompt = """
Ваша задача назначить других агентов для сотрудничества и решения данной задачи.
Имена агентов и их описания перечислены ниже:
{agents}
Данная задача:
{question}
Агенты обмениваются информацией через общую доску.
Основываясь на содержимом, которое уже есть на доске, вам необходимо выбрать подходящих агентов из списка, чтобы они оставили записи на доске.
"""

def create_controller_agent(role_agents : list[RoleAgent], board : Board):
    system_prompt = _controller_prompt.format(
        agents=[rl.info for rl in role_agents],
        question=board.question
    )
    return Agent(
        system_prompt=system_prompt,
        tools=[
            tool(board.get_board_notes),
            tool(board.get_board_note)  
        ],
        response_format=ControllerResponse,
    )

In [11]:
def _get_agent(agent_id : str, agents : list[Agent]) -> Agent | None:
    for agent in agents:
        if agent.id == agent_id:
            return agent
    return None

def get_agents(agents_ids : list[str], agents : list[Agent]):
    result = []
    for agent_id in agents_ids:
        agent = _get_agent(agent_id, agents)
        if agent is not None:
            result.append(agent)
    return result

### Wikipedia searcher

In [12]:
from typing import Literal
import re
import wikipedia
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from src import embedding

class PageSummary(BaseModel):
    id : int = Field(description='ID страницы')
    title : str = Field(description='Заголовок страницы')
    summary : str = Field(description='Краткое содержание страницы')

class WikipediaSearcher():

    def __init__(self, chunk_size : int = 1000, chunk_overlap : int = 100):
        self.stores = {}
        self.embedding = embedding
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

    def _get_page_docs(self, page_id):
        page = wikipedia.page(pageid=page_id, auto_suggest=False)
        splits = page.content.split('\n\n\n')
        path = [page.title]
        docs = []
        for split in splits:
            matches = re.findall(r'^(=+)\s*(.+?)\s*=+', split)
            if len(matches)>0:
                h, heading = matches[0]
                i = len(h) - 1
                path = [*path[:i], heading]
                split = split.removeprefix(f'{h} {heading} {h}')
            doc = Document(split, metadata={'path':path})
            if len(doc.page_content) > 0:
                docs.append(doc)
        return self.splitter.split_documents(docs)

    def _store_page(self, page_id : int) -> InMemoryVectorStore:
        if page_id in self.stores:
            return self.stores[page_id]
        docs = self._get_page_docs(page_id)
        store = InMemoryVectorStore.from_documents(docs, self.embedding)
        self.stores[page_id] = store
        return store 

    def search_on_wiki_page(self, query : str, page_id : int) -> list:
        """
        Возвращает результаты поиска по выбранной странице Википедии

        Args:
        - query : str - текстовый запрос
        - page_id : int - id страницы на Википедии

        Notes:
        - page_id может быть получен из PageSummary в search_for_wiki_pages()
        """
        try:
            store = self._store_page(page_id)
            return store.similarity_search(query, k=5)
        except:
            return []

    def search_for_wiki_pages(self, query : str, lang : Literal['ru', 'en'] = 'ru') -> list[PageSummary]:
        """
        Возвращает до 3 первых страниц на Википедии по указанному запросу на выбранном языке.
        
        Args:
        - query : str - текстовый запрос
        - lang : Literal['ru', 'en'] - выбранный язык Википедии (по умолчанию 'ru')
        """
        wikipedia.set_lang(lang)
        results = wikipedia.search(query, results=3)
        pages = []
        for r in results:
            try:
                page = wikipedia.page(title=r, auto_suggest=True)
                pages.append(page)
            except:
                pass
        unique_pages = {p.pageid:p for p in pages}
        return [PageSummary(
            id=up.pageid,
            title=up.title,
            summary=up.summary
        ) for up in unique_pages.values()]

In [13]:
_wikipedia_prompt = """
Вы {role_name}, сотрудничающий с другими агентами для решения задачи.
Задача:
{question}
Существует общая доска, которую каждый из вас может читать и на которой может оставлять записи.

Используя Википедию и основываясь на текущем содержимом общей доски, решите задачу, изложите свои идеи и информацию, которую вы хотите записать на доску.
Совершенно не обязательно полностью соглашаться с точкой зрения, представленной на доске.

Правила работы:
- Не вызывай один и тот же инструмент доступа к Википедии повторно с почти одинаковыми запросами, если только предыдущий вызов не вернул ноль полезных результатов.
- После одного или двух полезных вызовов инструментов доступа к Википедии остановись и верни итоговый ответ.
- Не используй общие знания. Отвечай на основе данных из Википедии.
"""

def create_wikipedia_agent(searcher : WikipediaSearcher, board : Board):
    role = Role(
        name='Вики-эксперт',
        description='Находит и обобщает информацию из Википедии'
    )
    system_prompt = _id_prompt + (_system_prompt + _wikipedia_prompt).format(
        role_name=role.name,
        question=board.question,
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_board_notes),
            tool(board.get_board_note),
            tool(searcher.search_for_wiki_pages),
            tool(searcher.search_on_wiki_page)
        ],
        response_format=BaseNote,
    )
    return agent

## Experiment

In [14]:
question = """
Разработка целеполагания г. Тюмень на 2026-2035 годы
"""

board = Board(question=question)

generator_agent = create_generator_agent(board)
roles = generator_agent.invoke([]).roles

In [15]:
expert_agents = [create_expert_agent(r, board) for r in roles]

for expert in expert_agents:
    print(expert.role.name, expert.role.description)

Стратегический урбанист‑планировщик Разрабатывает общую концепцию развития города, формирует долгосрочные пространственные стратегии и согласует их с муниципальными программами
Экономист регионального развития Проводит макроэкономический анализ, оценивает потенциал отраслей (нефть‑газ, производство, услуги), формирует финансово‑экономические цели и
Эксперт по инфраструктуре и транспортным системам Разрабатывает планы модернизации дорожной сети, общественного транспорта, логистических узлов и коммунальных сетей


In [16]:
searcher = WikipediaSearcher()

wikipedia_agent = create_wikipedia_agent(searcher, board)
decider_agent = create_decider_agent(board)
planner_agent = create_planner_agent(board)
critic_agent = create_critic_agent(board)
cleaner_agent = create_cleaner_agent(board)

role_agents = [planner_agent, wikipedia_agent, *expert_agents, critic_agent, cleaner_agent, decider_agent]
controller_agent = create_controller_agent(role_agents, board)

In [ ]:
from tqdm import tqdm
from langchain.messages import AIMessage, HumanMessage, SystemMessage


final_answer = None

for i in range(10):
    print(f'Итерация {i+1}')
    agents_ids = controller_agent.invoke(
        [SystemMessage(f'Записей на доске: {len(board.notes)}')], force=True
    ).agents_ids
    agents = get_agents(agents_ids, role_agents)
    print(str.join('\n', ['- ' + a.role.name for a in agents]))

    for a in agents:

        response = a.invoke([SystemMessage(f'Записей на доске: {len(board.notes)}')], force=True)
        if a == decider_agent:
            content = response.content
            if isinstance(content, BaseNote):
                note = content
            else:
                final_answer = content
            if final_answer is not None:
                break
        elif a == cleaner_agent:
            notes_ids = response.notes_ids
            for note_id in notes_ids:
                board.remove_note(note_id)
            note = response.note
        else:
            note = response
        note_id = board.add_note(note, a.id)
        print(f'{note_id}, {a.role.name}: {note.summary}')

    if final_answer is not None:
        break

Итерация 1
- Планировщик
- Вики-эксперт
- Стратегический урбанист‑планировщик
- Экономист регионального развития
- Эксперт по инфраструктуре и транспортным системам
d56efa, Планировщик: Created a detailed step‑by‑step plan for developing Tyumen’s 2026‑2035 goal‑setting strategy, including data collection, analysis, vision formulation, goal definition, program design, budgeting, stakeholder engagement, monitoring, and final documentation.
997e29, Вики-эксперт: Записал на доску аналитический обзор текущего состояния Тюмени (население, транспорт, роль в нефтегазовом регионе) и предложил структуру целеполагания на 2026‑2035 годы: SWOT‑анализ, ключевые направления (диверсификация экономики, развитие инфраструктуры, цифровизация, экологич‑н
1723eb, Стратегический урбанист‑планировщик: Предложена полная стратегическая концепция целеполагания Тюмени 2026‑2035 гг.: миссия, видение, 4 SMART‑цели (диверсификация экономики, устойчивый транспорт, экологическая безопасность, цифровая инфраструктура)

Deserializing unregistered type __main__.ControllerResponse from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ControllerResponse')]


7643d7, Эксперт по инфраструктуре и транспортным системам: Предложены инфраструктурные и транспортные цели для стратегии Тюмени 2026‑2035: оценка текущего состояния, план модернизации дорожной сети (реконструкция 300 км, кольцевая автодорога), переход к устойчивому общественному транспорту (электробусы, BRT‑коридоры, расширение трамвайки
Итерация 2


Deserializing unregistered type __main__.BaseNote from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'BaseNote')]


- Планировщик
- Вики-эксперт
- Стратегический урбанист‑планировщик
- Экономист регионального развития
- Эксперт по инфраструктуре и транспортным системам


...

Maybe tool calls or something idk


Deserializing unregistered type __main__.BaseNote from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'BaseNote')]


Maybe tool calls or something idk


Deserializing unregistered type __main__.BaseNote from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'BaseNote')]


Maybe tool calls or something idk


In [ ]:
print(final_answer)

На текущей доске присутствуют лишь краткие описания четырёх ключевых компонентов стратегии развития Тюмени — общего плана целеполагания, градостроительной концепции, экономического фундамента и финансового планирования, а также мета‑запись о полезности всех записей. Эти сведения дают представление о структуре будущей стратегии, но не содержат конкретных целей, показателей, сроков, приоритетных проектов и детализированных шагов реализации, которые требуются для окончательного формулирования целеполагания на 2026‑2035 годы.

**Вывод:** информации на доске недостаточно для завершения задачи. Необходимо получить более подробные материалы (конкретные SMART‑цели, KPI, дорожные карты, распределение инвестиций, оценку рисков и т.д.) от остальных участников.

**Рекомендация:** запросить у коллег дополнительные записи, содержащие:
1. Полный перечень стратегических целей (социальные, экономические, экологические, инфраструктурные) с количественными показателями.
2. Детализированные мероприятия и 

In [ ]:
board.notes

[Note(content='### Описание задачи\nРазработать стратегию целеполагания для города Тюмень на период 2026‑2035\u202fгг.\u202fЭто подразумевает формулирование долгосрочных целей развития (экономических, социальных, инфраструктурных, экологических и др.), определение приоритетных направлений, метрик и механизмов их достижения, а также согласование с интересами ключевых стейкхолдеров (власти, бизнес‑сообщества, граждан, научных и образовательных учреждений).\n\n### План решения задачи\n1. **Формирование рабочей группы**\n   - Определить роли: аналитик‑исследователь, специалист по стратегическому планированию, эксперт по городской инфраструктуре, представитель бизнеса, представитель гражданского общества, модератор.\n   - Согласовать сроки и каналы коммуникации (чат, видеоконференции, совместный документ).\n\n2. **Сбор и анализ исходных данных**\n   - Сбор статистики по демографии, экономике, бюджету, инфраструктуре, экологии Тюмени за последние 5‑10 лет.\n   - Анализ текущих стратегических